# Hodgkin-Huxley Model: Action Potential Generation

## Overview
The Hodgkin-Huxley model describes how action potentials in neurons are initiated and propagated. It's a system of coupled ODEs modeling voltage-gated ion channels.

## Mathematical Framework

### Membrane Voltage Equation:
$$C_m \dot{V} = -\bar{g}_{K} n^4 (V-E_{K}) - \bar{g}_{Na} m^3 h (V-E_{Na}) - g_L (V-E_L) + I$$

### Gating Variables:
$$\tau_n(V) \dot{n} = n_\infty(V)-n$$
$$\tau_m(V) \dot{m} = m_\infty(V)-m$$
$$\tau_h(V) \dot{h} = h_\infty(V)-h$$

where:
- $V$: membrane voltage (mV)
- $n$: potassium activation variable
- $m$: sodium activation variable
- $h$: sodium inactivation variable
- $I$: external current (µA/cm²)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)

## Model Parameters
Constants from the original Hodgkin-Huxley paper (1952)

In [ ]:
# Constants (parameters from original HH paper)
C_m = 1.0      # membrane capacitance (µF/cm²)
g_Na = 120.0   # sodium conductance (mS/cm²)
g_K = 36.0     # potassium conductance (mS/cm²)
g_L = 0.3      # leak conductance (mS/cm²)
E_Na = 50.0    # sodium reversal potential (mV)
E_K = -77.0    # potassium reversal potential (mV)
E_L = -54.387  # leak reversal potential (mV)

print("Hodgkin-Huxley Model Parameters:")
print(f"  Membrane capacitance: {C_m} µF/cm²")
print(f"  Na⁺ conductance: {g_Na} mS/cm²")
print(f"  K⁺ conductance: {g_K} mS/cm²")
print(f"  Leak conductance: {g_L} mS/cm²")

## Voltage-Dependent Rate Functions

These functions describe how gating variables respond to voltage changes

In [ ]:
# Voltage-dependent rate functions for sodium activation (m)
def alpha_m(V):
    """Opening rate for sodium activation gate"""
    return 0.1 * (V + 40.0) / (1.0 - np.exp(-(V + 40.0) / 10.0))

def beta_m(V):
    """Closing rate for sodium activation gate"""
    return 4.0 * np.exp(-(V + 65.0) / 18.0)

# Voltage-dependent rate functions for sodium inactivation (h)
def alpha_h(V):
    """Opening rate for sodium inactivation gate"""
    return 0.07 * np.exp(-(V + 65.0) / 20.0)

def beta_h(V):
    """Closing rate for sodium inactivation gate"""
    return 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))

# Voltage-dependent rate functions for potassium activation (n)
def alpha_n(V):
    """Opening rate for potassium activation gate"""
    return 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0))

def beta_n(V):
    """Closing rate for potassium activation gate"""
    return 0.125 * np.exp(-(V + 65.0) / 80.0)

## Ion Currents
Each current is proportional to conductance, gating variables, and driving force

In [ ]:
# Ion currents
def I_Na(V, m, h):
    """Sodium current"""
    return g_Na * m**3 * h * (V - E_Na)

def I_K(V, n):
    """Potassium current"""
    return g_K * n**4 * (V - E_K)

def I_L(V):
    """Leak current"""
    return g_L * (V - E_L)

## External Stimulus
Define the current injection protocol

In [ ]:
# External current (stimulus)
def I_ext(t, I_amplitude=10.0, t_start=10.0, t_end=40.0):
    """External current stimulus"""
    if t_start <= t <= t_end:
        return I_amplitude
    else:
        return 0.0

## System of ODEs
Combined equations for voltage and gating variables

In [ ]:
def hodgkin_huxley(y, t):
    """
    Hodgkin-Huxley ODEs
    
    Parameters:
    y: [V, m, h, n] - state variables
    t: time
    
    Returns:
    derivatives: [dV/dt, dm/dt, dh/dt, dn/dt]
    """
    V, m, h, n = y
    
    # Membrane voltage equation
    dVdt = (I_ext(t) - I_Na(V, m, h) - I_K(V, n) - I_L(V)) / C_m
    
    # Gating variable equations
    dmdt = alpha_m(V) * (1.0 - m) - beta_m(V) * m
    dhdt = alpha_h(V) * (1.0 - h) - beta_h(V) * h
    dndt = alpha_n(V) * (1.0 - n) - beta_n(V) * n
    
    return [dVdt, dmdt, dhdt, dndt]

## Initial Conditions
Set the neuron at resting state

In [ ]:
# Initial conditions (resting state)
V0 = -65.0  # initial membrane potential (mV)
m0 = alpha_m(V0) / (alpha_m(V0) + beta_m(V0))
h0 = alpha_h(V0) / (alpha_h(V0) + beta_h(V0))
n0 = alpha_n(V0) / (alpha_n(V0) + beta_n(V0))
y0 = [V0, m0, h0, n0]

print("Initial conditions (resting state):")
print(f"  V₀ = {V0:.2f} mV")
print(f"  m₀ = {m0:.4f}")
print(f"  h₀ = {h0:.4f}")
print(f"  n₀ = {n0:.4f}")

## Numerical Integration
Solve the system of ODEs using scipy's odeint

In [ ]:
# Time vector
t = np.linspace(0, 100, 10000)  # 100 ms simulation

# Solve ODEs
print("Integrating Hodgkin-Huxley equations...")
solution = odeint(hodgkin_huxley, y0, t)

# Extract variables
V = solution[:, 0]
m = solution[:, 1]
h = solution[:, 2]
n = solution[:, 3]

print("Integration complete!")

## Visualization: Action Potential
Plot membrane voltage and gating variables

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Plot 1: Membrane voltage
axes[0].plot(t, V, 'k', linewidth=2)
axes[0].set_ylabel('Voltage (mV)', fontsize=12)
axes[0].set_title('Hodgkin-Huxley Model: Action Potential', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='0 mV')
axes[0].legend()

# Plot 2: Gating variables
axes[1].plot(t, m, 'r', label='m (Na⁺ activation)', linewidth=2)
axes[1].plot(t, h, 'b', label='h (Na⁺ inactivation)', linewidth=2)
axes[1].plot(t, n, 'g', label='n (K⁺ activation)', linewidth=2)
axes[1].set_ylabel('Gating Variable', fontsize=12)
axes[1].set_ylim([0, 1])
axes[1].legend(loc='right', fontsize=10)
axes[1].grid(True, alpha=0.3)

# Plot 3: Ionic currents
I_Na_arr = np.array([I_Na(V[i], m[i], h[i]) for i in range(len(t))])
I_K_arr = np.array([I_K(V[i], n[i]) for i in range(len(t))])
I_L_arr = np.array([I_L(V[i]) for i in range(len(t))])

axes[2].plot(t, I_Na_arr, 'r', label='I_Na', linewidth=2)
axes[2].plot(t, I_K_arr, 'g', label='I_K', linewidth=2)
axes[2].plot(t, I_L_arr, 'gray', label='I_L', linewidth=1, alpha=0.7)
axes[2].set_xlabel('Time (ms)', fontsize=12)
axes[2].set_ylabel('Current (µA/cm²)', fontsize=12)
axes[2].legend(loc='upper right', fontsize=10)
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('/home/user/hh_action_potential.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAction potential characteristics:")
print(f"  Peak voltage: {V.max():.2f} mV")
print(f"  Resting voltage: {V[0]:.2f} mV")
print(f"  Spike amplitude: {V.max() - V[0]:.2f} mV")

## Phase Space Analysis
Visualize the system in 2D phase space (V vs n)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Phase trajectory
ax.plot(V, n, 'b-', linewidth=2, alpha=0.7)
ax.plot(V[0], n[0], 'go', markersize=10, label='Start (rest)')
ax.plot(V[-1], n[-1], 'ro', markersize=10, label='End')

# Mark spike peak
peak_idx = np.argmax(V)
ax.plot(V[peak_idx], n[peak_idx], 'r*', markersize=15, label='Spike peak')

ax.set_xlabel('Voltage V (mV)', fontsize=12)
ax.set_ylabel('K⁺ activation n', fontsize=12)
ax.set_title('Phase Space: V-n Trajectory', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/user/hh_phase_space.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercise: Explore Different Stimuli
Modify the stimulus parameters and observe changes in firing behavior

In [ ]:
# Test different current amplitudes
I_amplitudes = [5, 10, 15, 20]

fig, axes = plt.subplots(len(I_amplitudes), 1, figsize=(12, 10), sharex=True)

for i, I_amp in enumerate(I_amplitudes):
    # Redefine I_ext with new amplitude
    def I_ext_temp(t):
        return I_ext(t, I_amplitude=I_amp)
    
    # Create new ODE function
    def hh_temp(y, t):
        V, m, h, n = y
        dVdt = (I_ext_temp(t) - I_Na(V, m, h) - I_K(V, n) - I_L(V)) / C_m
        dmdt = alpha_m(V) * (1.0 - m) - beta_m(V) * m
        dhdt = alpha_h(V) * (1.0 - h) - beta_h(V) * h
        dndt = alpha_n(V) * (1.0 - n) - beta_n(V) * n
        return [dVdt, dmdt, dhdt, dndt]
    
    # Solve
    sol = odeint(hh_temp, y0, t)
    V_temp = sol[:, 0]
    
    # Plot
    axes[i].plot(t, V_temp, 'k', linewidth=2)
    axes[i].set_ylabel('V (mV)', fontsize=10)
    axes[i].set_title(f'I = {I_amp} µA/cm²', fontsize=11)
    axes[i].grid(True, alpha=0.3)
    axes[i].axhline(y=0, color='r', linestyle='--', alpha=0.3)

axes[-1].set_xlabel('Time (ms)', fontsize=12)
fig.suptitle('Response to Different Current Amplitudes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/user/hh_different_currents.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **System of ODEs**: The HH model consists of 4 coupled differential equations
2. **Gating Variables**: Ion channels have voltage-dependent open/close kinetics
3. **Ionic Currents**: Different ions (Na⁺, K⁺) create the action potential shape
4. **Numerical Integration**: `scipy.integrate.odeint` solves the system efficiently
5. **Phase Space**: Reveals the limit cycle behavior of action potentials

## Further Exploration

- Change conductance values (g_Na, g_K, g_L)
- Modify reversal potentials (E_Na, E_K, E_L)
- Try different stimulus durations and amplitudes
- Add more ionic currents (Ca²⁺, T-type channels, etc.)